### 1. Load the Dataset
Load the input CSV file into a pandas DataFrame using pd.read_csv

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('dirty_cafe_sales.csv')

-------------

### 2. Exploring the Data
we should explor the data before doing anything

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

----------

### 3. Rename the columns
Rename the columns using rename in pandas

In [ ]:
df.info()

In [ ]:
df = df.rename(columns={
    "Transaction ID" : "Transaction_ID",
    "Price Per Unit" : "Price_Per_Unit",
    "Total Spent" : "Total_Spent",
    "Payment Method" : "Payment_Method",
    "Transaction Date" : "Transaction_Date"
})

---------------

### 4. Handle missing and invalid strings
replace the placeholder strings "ERROR" and "UNKNOWN" with np.nan

In [ ]:
df = df.replace({
    'ERROR' : np.nan,
    'UNKNOWN': np.nan
})

---------------

### 5. Data Type Correction 
numerical columns to numeric arrays and dates to datetime

In [ ]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price_Per_Unit'] = pd.to_numeric(df['Price_Per_Unit'], errors='coerce')
df['Total_Spent'] = pd.to_numeric(df['Total_Spent'], errors='coerce')

df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors='coerce')

---------------

### 6. Data Consistency and Integrity
We will fill in missing information based on logic:
1. Missing Item names dynamically by finding the most frequent item Mode for that price
2. restore missing Price_Per_Unit for known items using their fixed price
3. Missing Total_Spent or Quantity can be calculated from Total_Spent = Quantity * Price_Per_Unit

In [ ]:
def get_first_mode(x):
    m = x.mode()
    if not m.empty:
        return m.iloc[0]
    return np.nan

In [ ]:
mode_per_price = df.groupby('Price_Per_Unit')['Item'].agg(get_first_mode)
df['Item'] = df['Item'].fillna(df['Price_Per_Unit'].map(mode_per_price))

In [ ]:
price_per_item = df.groupby('Item')['Price_Per_Unit'].agg(get_first_mode)
mask_price = df['Price_Per_Unit'].isna() & df['Item'].notna()
df.loc[mask_price, 'Price_Per_Unit'] = df.loc[mask_price, 'Item'].map(price_per_item)

In [ ]:
mask_total = df['Total_Spent'].isna() & df['Quantity'].notna() & df['Price_Per_Unit'].notna()
df.loc[mask_total, 'Total_Spent'] = df.loc[mask_total, 'Quantity'] * df.loc[mask_total, 'Price_Per_Unit']

In [ ]:
mask_qty = df['Quantity'].isna() & df['Total_Spent'].notna() & df['Price_Per_Unit'].notna()
df.loc[mask_qty, 'Quantity'] = df.loc[mask_qty, 'Total_Spent'] / df.loc[mask_qty, 'Price_Per_Unit']
df['Quantity'] = df['Quantity'].round()

In [ ]:
mask_price2 = df['Price_Per_Unit'].isna() & df['Quantity'].notna() & df['Total_Spent'].notna()
df.loc[mask_price2, 'Price_Per_Unit'] = df.loc[mask_price2, 'Total_Spent'] / df.loc[mask_price2, 'Quantity']

In [ ]:
df.head()

----------------------

### 7. Final handling of Missing Values
Now we drop rows where:
1. numerical columns cannot be recovered because they represent fundamentally invalid transactions
2. there is no proper date for the same reason

For categorical columns: 
1. We fill them with "Unknown" so they can safely remain in the dataset

In [ ]:
df = df.dropna(subset=['Quantity', 'Price_Per_Unit', 'Total_Spent', 'Transaction_Date'])

df['Item'] = df['Item'].fillna('Unknown')
df['Payment_Method'] = df['Payment_Method'].fillna('Unknown')
df['Location'] = df['Location'].fillna('Unknown')

------------

### 8. Create Season
We use the Transaction Date to extract the month and assign a season

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

In [ ]:
df['season'] = df['Transaction_Date'].dt.month.apply(get_season)
df.head()

-----------

### 9. Output Cleaned Data
Save the cleaned DataFrame to cleaned_cafe_sales.csv

In [ ]:
df.to_csv('cleaned_cafe_sales.csv', index=False)

--------------